# Clinical Strand — Model Training

**MultimodalAI‧26 · Clinical Demo**

---

Trains **Model A** (LightGBM, all tabular features) and **Model C** (PyKale multimodal
fusion: tabular + clinical notes text + hourly vitals time series).

**Prerequisites:**
- [ ] `data/processed/train.csv`, `val.csv`, `test.csv` — run `01_preprocess_and_split.ipynb`
- [ ] `data/raw/notes.csv`, `vitals_series.csv` — run `setup.py`

**Sections**
1. Setup & Load Data
2. Train Model A — LightGBM baseline
3. Train Model C — PyKale multimodal fusion (notes MNAR preview)
4. Calibration Check
5. Save Models and Metrics
6. Launch App

---
## Section 1 — Setup & Load Data

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / 'models').exists() and (candidate / 'src').exists() and (candidate / 'data').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate clinical_strand root.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROC_DIR   = ROOT / 'data' / 'processed'
RAW_DIR    = ROOT / 'data' / 'raw'
MODEL_DIR  = ROOT / 'saved_models'
METRIC_DIR = ROOT / 'saved_metrics'
TARGET     = 'major_complication_30d'

for d in [MODEL_DIR, METRIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')

In [ ]:
# Load processed splits
for fname in ['train.csv', 'val.csv', 'test.csv']:
    if not (PROC_DIR / fname).exists():
        raise FileNotFoundError(f'{fname} not found. Run 01_preprocess_and_split.ipynb first.')

df_train = pd.read_csv(PROC_DIR / 'train.csv')
df_val   = pd.read_csv(PROC_DIR / 'val.csv')
df_test  = pd.read_csv(PROC_DIR / 'test.csv')

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

print('Splits loaded:')
for split, name in [(df_train, 'train'), (df_val, 'val'), (df_test, 'test')]:
    print(f'  {name:6s}: n={len(split):4d}  complication={split[TARGET].mean():.1%}')

In [ ]:
# Load notes and vitals — needed for Model C
notes_df  = pd.read_csv(RAW_DIR / 'notes.csv')
vitals_df = pd.read_csv(RAW_DIR / 'vitals_series.csv')

# Align modality files to each split by patient_id
train_notes  = notes_df[notes_df['patient_id'].isin(df_train['patient_id'])].reset_index(drop=True)
val_notes    = notes_df[notes_df['patient_id'].isin(df_val['patient_id'])].reset_index(drop=True)
test_notes   = notes_df[notes_df['patient_id'].isin(df_test['patient_id'])].reset_index(drop=True)

train_vitals = vitals_df[vitals_df['patient_id'].isin(df_train['patient_id'])].reset_index(drop=True)
val_vitals   = vitals_df[vitals_df['patient_id'].isin(df_val['patient_id'])].reset_index(drop=True)
test_vitals  = vitals_df[vitals_df['patient_id'].isin(df_test['patient_id'])].reset_index(drop=True)

print(f'Notes loaded  : {len(notes_df)} records (patients with has_notes == 1)')
print(f'Vitals loaded : {len(vitals_df)} records (24 rows per patient)')

In [ ]:
from models import ModelA
from models.model_c import ModelC

# Check if evaluate.py is available for richer metrics
_EVAL_READY = False
try:
    from src.evaluate import compute_metrics, compute_calibration, save_metrics_csv
    _test = compute_metrics(np.array([0, 1, 0, 1]), np.array([0.1, 0.9, 0.2, 0.8]))
    assert 'auroc' in _test
    _EVAL_READY = True
    print('src/evaluate.py: ready')
except Exception as e:
    print(f'src/evaluate.py: {e}')

---
## Section 2 — Train Model A (LightGBM baseline)

**Model A** is a LightGBM classifier using all available tabular features:
demographics, surgical context, ICU severity, vitals aggregates, labs, and notes flag.
It is the performance ceiling — strongest overall as it sees the most information.

**Known limitation:** `blood_loss_imputed` uses mean imputation when blood loss is
not recorded (MNAR ~36%). True blood loss is higher for missing cases — the model
systematically underestimates risk for cardiac/vascular patients.

In [ ]:
model_a = ModelA()
model_a.fit(df_train, y_train)

prob_a_val = model_a.predict_proba(df_val)[:, 1]

if _EVAL_READY:
    m_a = compute_metrics(y_val, prob_a_val, model_name='Model A')
    print(f'Model A | Val AUROC={m_a["auroc"]:.3f}  '
          f'Sensitivity={m_a["sensitivity"]:.3f}  '
          f'Brier={m_a["brier_score"]:.3f}')
else:
    from sklearn.metrics import roc_auc_score
    print(f'Model A | Val AUROC={roc_auc_score(y_val, prob_a_val):.3f}')

In [ ]:
fi_a = model_a.feature_importance().head(10)
fig, ax = plt.subplots(figsize=(8, 4))
fi_a.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Model A — Top 10 feature importances')
ax.set_xlabel('Importance score')
plt.tight_layout(); plt.show()

---
## Section 3 — Train Model C (PyKale Multimodal Fusion)

**Model C** is a three-modality fusion network (PyKale 0.2.0):
- **Tabular branch**: surgical context + ICU severity (FCNet encoder)
- **Text branch**: TF-IDF on free-text clinical notes (FCNet encoder)
- **Time series branch**: slope/variability features from hourly vitals (FCNet encoder)

**Known limitation:** when notes are absent (MNAR ~32%), the text branch receives
a zero vector, causing systematic underestimation of risk for note-absent patients —
concentrated in complex cardiac/vascular cases.

Training takes ~30–60 seconds on CPU.

In [ ]:
model_c = ModelC()
model_c.fit(df_train, y_train, notes_df=train_notes, vitals_df=train_vitals)

prob_c_val = model_c.predict_proba_full(df_val, val_notes, val_vitals)[:, 1]

if _EVAL_READY:
    m_c = compute_metrics(y_val, prob_c_val, model_name='Model C')
    print(f'Model C | Val AUROC={m_c["auroc"]:.3f}  '
          f'Sensitivity={m_c["sensitivity"]:.3f}  '
          f'Brier={m_c["brier_score"]:.3f}')
else:
    from sklearn.metrics import roc_auc_score
    print(f'Model C | Val AUROC={roc_auc_score(y_val, prob_c_val):.3f}')

In [ ]:
# MNAR preview: compare Model C performance for patients with vs without notes
if _EVAL_READY:
    print('Model C MNAR preview — notes subgroup:')
    for has, label in [(1, 'With notes   '), (0, 'Without notes')]:
        mask = df_val['has_notes'].values == has
        if mask.sum() < 5:
            continue
        m = compute_metrics(y_val[mask], prob_c_val[mask])
        print(f'  {label}: AUROC={m["auroc"]:.3f}  Sensitivity={m["sensitivity"]:.3f}  n={m["n_total"]}')

    no_mask  = df_val['has_notes'].values == 0
    yes_mask = df_val['has_notes'].values == 1
    if no_mask.sum() > 5 and yes_mask.sum() > 5:
        gap = (compute_metrics(y_val[yes_mask], prob_c_val[yes_mask])['auroc'] -
               compute_metrics(y_val[no_mask],  prob_c_val[no_mask])['auroc'])
        print(f'  AUROC gap (with minus without notes): {gap:.3f}')
        if gap > 0.05:
            print('  >>> MNAR signal detected. Report this in Tabs 3 and 5 of the app.')
else:
    print('MNAR preview skipped — evaluate.py not available.')

---
## Section 4 — Calibration Check

A well-calibrated model is one where predicted probabilities match observed outcomes.
If Model A predicts 70% risk, those patients should actually complicate ~70% of the time.
Poor calibration undermines the ward simulation in Tab 2 of the app.

In [ ]:
if _EVAL_READY:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, prob, name, colour in [
        (axes[0], prob_a_val, 'Model A', 'steelblue'),
        (axes[1], prob_c_val, 'Model C', 'seagreen'),
    ]:
        cal = compute_calibration(y_val, prob, n_bins=8)
        ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=1, label='Perfect')
        sc = ax.scatter(cal['mean_predicted'], cal['fraction_positive'],
                        c=cal['bin_counts'], cmap='Blues', s=80, zorder=5)
        ax.plot(cal['mean_predicted'], cal['fraction_positive'],
                color=colour, linewidth=1.5, alpha=0.7)
        plt.colorbar(sc, ax=ax, label='N patients')
        ax.set_title(f'{name} — calibration')
        ax.set_xlabel('Mean predicted prob')
        ax.set_ylabel('Observed complication rate')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    plt.suptitle('Calibration (val set) — on diagonal = well calibrated', fontsize=11)
    plt.tight_layout(); plt.show()
    print('Above diagonal = model under-predicts risk')
    print('Below diagonal = model over-predicts risk')
else:
    print('Calibration plot skipped — evaluate.py not available.')

---
## Section 5 — Save Models and Metrics

In [ ]:
joblib.dump(model_a, MODEL_DIR / 'model_a.joblib')
joblib.dump(model_c, MODEL_DIR / 'model_c.joblib')
print(f'Models saved to {MODEL_DIR}/')

In [ ]:
if _EVAL_READY:
    val_results = [
        compute_metrics(y_val, prob_a_val, model_name='Model A'),
        compute_metrics(y_val, prob_c_val, model_name='Model C'),
    ]
    summary_cols = ['model', 'auroc', 'auprc', 'sensitivity', 'specificity', 'brier_score']
    print('Validation summary:')
    print(pd.DataFrame(val_results)[summary_cols].to_string(index=False))
    save_metrics_csv(val_results, str(METRIC_DIR / 'training_metrics.csv'))
    print('\nSaved training_metrics.csv')
else:
    from sklearn.metrics import roc_auc_score
    print(f'Model A AUROC: {roc_auc_score(y_val, prob_a_val):.3f}')
    print(f'Model C AUROC: {roc_auc_score(y_val, prob_c_val):.3f}')

---
## Section 6 — Launch App

Models and metrics are saved. Launch the evaluation app:

```bash
cd ..    # back to clinical_strand/
streamlit run app.py
```

| Tab | What to examine |
|---|---|
| Tab 1 — Step-Down Brief | Per-patient risk scores, Model A vs C consensus |
| Tab 2 — Model Performance | AUROC, calibration, ward simulation |
| Tab 3 — Equity & MNAR | Surgery type subgroups; notes-absent AUROC gap for Model C |
| Tab 4 — Failure Modes | High-risk patients missed (false negatives) |
| Tab 5 — Report Builder | Deployment verdicts + Model Safety Report |

In [ ]:
print('Training complete.')
if _EVAL_READY and 'val_results' in dir():
    for r in val_results:
        print(f'  {r["model"]}: AUROC={r["auroc"]:.3f}')
print()
print('Launch app: streamlit run app.py')